In [1]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt


In [2]:
df_train = pd.read_csv("Data/archive/fashion-mnist_train.csv")
df_test = pd.read_csv("Data/archive/fashion-mnist_test.csv")

In [3]:
class FashionDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [4]:
# Prepare full dataset
X_full_train = df_train.iloc[:, 1:].values / 255.0
y_full_train = df_train.iloc[:, 0].values

X_full_test = df_test.iloc[:, 1:].values / 255.0
y_full_test = df_test.iloc[:, 0].values


In [5]:
full_train_dataset = FashionDataset(X_full_train, y_full_train)
full_test_dataset = FashionDataset(X_full_test, y_full_test)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [7]:
class MyNN(nn.Module):
    def __init__(self, input_dim, output_dim, num_layers, neurons_per_layer, dropout_rate):
        super().__init__()
        layers = []

        for _ in range(num_layers):
            layers.append(nn.Linear(input_dim, neurons_per_layer))
            layers.append(nn.BatchNorm1d(neurons_per_layer))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            input_dim = neurons_per_layer

        layers.append(nn.Linear(input_dim, output_dim))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [8]:
def objective(trial):
    
    # next hyperparameter value for search space
    num_layers = trial.suggest_int("num_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 32, 256, step=32)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5, step=0.1)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-1, log=True)
    epochs = trial.suggest_int("epochs", 10, 100, step=10)
    batch_size = trial.suggest_categorical("batch_size", [16,32,64,128])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD","RMSProp"])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    
    full_train_dataset_loader = DataLoader(full_train_dataset, batch_size=batch_size, shuffle=True,pin_memory=True)
    full_test_dataset_loader = DataLoader(full_test_dataset, batch_size=batch_size, shuffle=False)
    
    input_dim = X_full_train.shape[1]
    output_dim = len(set(y_full_train))
    # model init 
    model = MyNN(input_dim, output_dim, num_layers, neurons_per_layer, dropout_rate).to(device)
    
    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == "RMSProp":
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    # training loop
    for epoch in range(epochs):
        
        model.train()
        for batch_features, labels in full_train_dataset_loader:
            batch_features, labels = batch_features.float().to(device), labels.to(device)
            outputs = model(batch_features)
            loss = criterion(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
    # evaluation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_features, labels in full_test_dataset_loader:
            batch_features, labels = batch_features.float().to(device), labels.to(device)
            ypred = model(batch_features)
            _, predicted = torch.max(ypred, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = correct / total
    return accuracy

In [ ]:
import optuna 
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

In [28]:
study.get_trials()    

[FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.8535], datetime_start=datetime.datetime(2026, 3, 2, 0, 40, 30, 291662), datetime_complete=datetime.datetime(2026, 3, 2, 0, 41, 38, 841123), params={'num_layers': 2, 'neurons_per_layer': 224, 'dropout_rate': 0.1, 'learning_rate': 0.00034058427591722386, 'epochs': 20, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.0002641109519152009}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_layers': IntDistribution(high=5, log=False, low=1, step=1), 'neurons_per_layer': IntDistribution(high=256, log=False, low=32, step=32), 'dropout_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.1), 'learning_rate': FloatDistribution(high=0.1, log=True, low=0.0001, step=None), 'epochs': IntDistribution(high=100, log=False, low=10, step=10), 'batch_size': CategoricalDistribution(choices=(16, 32, 64, 128)), 'optimizer': CategoricalDistribution(choices=('Adam', 'SGD', 'RMSProp')), 'weight_decay': Flo

In [31]:
study.best_value

0.8535

In [ ]:
study.best_params

In [36]:
study._storage.get_best_trial(study._study_id)

FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.8535], datetime_start=datetime.datetime(2026, 3, 2, 0, 40, 30, 291662), datetime_complete=datetime.datetime(2026, 3, 2, 0, 41, 38, 841123), params={'num_layers': 2, 'neurons_per_layer': 224, 'dropout_rate': 0.1, 'learning_rate': 0.00034058427591722386, 'epochs': 20, 'batch_size': 64, 'optimizer': 'SGD', 'weight_decay': 0.0002641109519152009}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_layers': IntDistribution(high=5, log=False, low=1, step=1), 'neurons_per_layer': IntDistribution(high=256, log=False, low=32, step=32), 'dropout_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.1), 'learning_rate': FloatDistribution(high=0.1, log=True, low=0.0001, step=None), 'epochs': IntDistribution(high=100, log=False, low=10, step=10), 'batch_size': CategoricalDistribution(choices=(16, 32, 64, 128)), 'optimizer': CategoricalDistribution(choices=('Adam', 'SGD', 'RMSProp')), 'weight_decay': Floa